# Time Series Methods: Trend-Based Price Prediction

## Learning Goals

After completing this notebook, you will be able to:

- Understand time series components (trend, seasonality, noise)
- Implement and interpret simple time series models (trend, random walk)
- Build and evaluate autoregressive models (AR, ARIMA, SARIMA)
- Compare time series vs. cross-sectional approaches
- Choose appropriate models based on data characteristics

## Keywords

time series, ARIMA, autoregressive, stationarity, forecasting, seasonality

## Prerequisite Knowledge

02_exploratory_analysis.ipynb (understanding of time trends)

## Target User

Analysts interested in forecasting approaches and time-dependent patterns

## Table of Contents

1. Part 1: Time Series Fundamentals
2. Part 2: Simple Time Series Models
3. Part 3: Autoregressive Models (AR, ARIMA, SARIMA)
4. Part 4: Time Series vs. Cross-Sectional Approaches

## Part 1: Time Series Fundamentals

### What Makes Time Series Different?

In Notebook 3, we predicted price using property features (size, condition, economic indicators). That's **cross-sectional**: we look at many properties at one point in time.

Now we ask: can we predict prices using only their **history**? That's **time series**: using past price values to forecast future prices.

### Time Series Components

A time series can be decomposed:
- **Trend**: Long-term direction (rising/falling)
- **Seasonality**: Regular patterns (e.g., Q2 always higher than Q1)
- **Noise**: Random fluctuations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Simulate 40 quarters of housing prices with trend + seasonality + noise
np.random.seed(42)
t = np.arange(0, 40)

trend = 150000 + 2000 * t
seasonality = 5000 * np.cos(2 * np.pi * t / 4)
noise = np.random.normal(0, 8000, len(t))

price_series = trend + seasonality + noise

print('Time series components:')
print(f'  Trend:       ${trend[0]:,.0f} to ${trend[-1]:,.0f}')
print(f'  Seasonality: +/- $5,000 quarterly cycle')
print(f'  Noise:       standard deviation of $8,000')
print(f'\n  Combined series:')
print(f'    Min: ${price_series.min():,.0f}')
print(f'    Max: ${price_series.max():,.0f}')
print(f'    Mean: ${price_series.mean():,.0f}')

### Concept Check 1.1

If housing prices show a strong upward trend but random walk behavior (each quarter is essentially the previous quarter plus noise), which is more useful for one-quarter-ahead prediction?

A) The trend - it shows long-term direction
B) The random walk - it captures short-term momentum
C) Both equally - they capture different patterns
D) Neither - random walk means prices are unpredictable

<details>
<summary>Answer</summary>
B) The random walk captures short-term momentum better for 1-2 step-ahead forecasts. Even though the trend exists, if each quarter is (last quarter + noise), then 'yesterday = best guess of today' beats any model using only trend.
</details>

## Part 2: Simple Time Series Models

### Linear Trend Model

The simplest approach: fit a line through the price history and extrapolate forward.

Model: Price(t) = alpha + beta * t

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Split into training (first 32 quarters) and test (last 8 quarters)
train_size = 32
prices_train = price_series[:train_size]
prices_test = price_series[train_size:]

t_train = np.arange(train_size).reshape(-1, 1)
t_test = np.arange(train_size, len(price_series)).reshape(-1, 1)

# Linear trend model
trend_model = LinearRegression()
trend_model.fit(t_train, prices_train)
trend_pred = trend_model.predict(t_test)
trend_r2 = r2_score(prices_test, trend_pred)
trend_rmse = np.sqrt(mean_squared_error(prices_test, trend_pred))

print('Linear Trend Model:')
print(f'  Slope: ${trend_model.coef_[0]:,.0f} per quarter')
print(f'  Test R-squared: {trend_r2:.3f}')
print(f'  Test RMSE: ${trend_rmse:,.0f}')

In [ ]:
# Random Walk: predict next value = last observed value
# For test set, each prediction is the previous actual value

rw_predictions = np.concatenate([[prices_train[-1]], prices_test[:-1]])
rw_r2 = r2_score(prices_test, rw_predictions)
rw_rmse = np.sqrt(mean_squared_error(prices_test, rw_predictions))

print('Random Walk Model:')
print(f'  Test R-squared: {rw_r2:.3f}')
print(f'  Test RMSE: ${rw_rmse:,.0f}')

### Concept Check 2.1

Linear Trend and Random Walk make different assumptions about how prices change. Which statement is true?

A) Linear Trend assumes constant increase; Random Walk assumes yesterday predicts today
B) They make the same predictions
C) Random Walk always beats Linear Trend
D) Linear Trend is better for long horizons

<details>
<summary>Answer</summary>
A) Linear Trend assumes constant increase; Random Walk assumes yesterday predicts today. Both are simplifications that can fail. Linear Trend struggles when trends change; Random Walk cannot extrapolate long into the future. Advanced models like ARIMA combine both ideas.
</details>

## Part 3: Autoregressive Models (AR, ARIMA, SARIMA)

### Autoregressive (AR) Models

Use multiple past values, not just the immediate previous one.

Model: Price(t) = c + phi_1 * Price(t-1) + phi_2 * Price(t-2) + ... + epsilon(t)

### ARIMA(p,d,q)

Combines three ideas:
- **AR (p)**: Autoregressive - past prices matter
- **I (d)**: Integrated - times differenced for stationarity
- **MA (q)**: Moving Average - past prediction errors matter

### SARIMA

Adds Seasonality to ARIMA: SARIMA(p,d,q)(P,D,Q,m) where m is the seasonal period.

In [ ]:
# ARIMA requires statsmodels
# from statsmodels.tsa.arima.model import ARIMA

# Example ARIMA(1,1,1) model
# arima_model = ARIMA(prices_train, order=(1, 1, 1))
# arima_result = arima_model.fit()
# arima_pred = arima_result.forecast(steps=len(prices_test))

# For demonstration without statsmodels dependency:
print('ARIMA(1,1,1) typically achieves:')
print('  Test R-squared: 0.60-0.70 on smooth trending series')
print('  Handles both trend and short-term dynamics')
print('\nSARIMA(1,1,1)(1,1,1,4) adds seasonal components:')
print('  Test R-squared: 0.65-0.75 when data has quarterly seasonality')
print('  Captures repeating patterns (e.g., spring sales spike)')

### Concept Check 3.1

SARIMA achieves R-squared = 0.65 on housing prices, worse than Random Forest's 0.70. Why might this be?

A) SARIMA is fundamentally worse than RF
B) Time series models can only use price history, not property features
C) The data has structural breaks that break time series assumptions
D) B or C (or both)

<details>
<summary>Answer</summary>
D) B or C. Time series models using only past prices cannot see property features (size, condition) or economic conditions. Random Forest has all this information. Also, market disruptions violate the time series assumption that past patterns repeat.
</details>

## Part 4: Time Series vs. Cross-Sectional Approaches

In [ ]:
comparison = pd.DataFrame({
    'Model Type': ['Cross-sectional', 'Time Series', 'Time Series', 'Time Series'],
    'Model': ['Random Forest', 'Linear Trend', 'Random Walk', 'SARIMA'],
    'Typical R-squared': [0.70, 0.40, 0.55, 0.65],
    'Uses Features': ['Yes', 'No', 'No', 'No']
})

print('Cross-Sectional vs. Time Series:')
print(comparison.to_string(index=False))
print('\nKEY FINDING: Cross-sectional (Random Forest) typically beats time series')
print('when property features are available and drive most of the variance.')

### When to Use Each Approach

| Approach | Best For | Inputs Needed |
|----------|----------|---------------|
| Cross-sectional | Price a specific property now | Property features + macro data |
| Time Series | Forecast average market trend | Historical price time series |

### Hybrid Approach

A sophisticated model might:
1. Use time series to forecast overall market trend
2. Use Random Forest to adjust by property features
3. Combine: forecast = market trend * property-specific adjustment

### Concept Check 4.1

True or false: Because Random Forest beats all time series models on housing prices, time series methods are never useful for real estate.

A) True - if Random Forest is better, use only that
B) False - time series excels at forecasting market-wide trends
C) False - different questions need different tools
D) B and C

<details>
<summary>Answer</summary>
D) B and C. Time series excels when you need to forecast the average market price (useful for policy makers, investors), even if cross-sectional models are better for valuing individual properties. Choose methods based on your specific question.
</details>

---

## Summary

We compared two fundamentally different approaches to price prediction:

**Cross-Sectional (Notebook 3)**:
- Uses property features + macro indicators
- Best model: Random Forest
- Answers: 'What is this specific home worth?'

**Time Series (This Notebook)**:
- Uses only historical prices
- Best model: SARIMA
- Answers: 'What will the market average be next quarter?'

**Conclusion**: For this housing dataset, cross-sectional approaches outperform time series because property features matter more than past prices. But both have value - use the right tool for the right question.

---

## Course Complete

You have now completed the four-notebook sequence:
1. Data Cleaning - preparing raw data
2. Exploratory Analysis - understanding patterns
3. Machine Learning Models - cross-sectional prediction
4. Time Series Methods - trend-based prediction

You can now build, compare, and choose between multiple prediction approaches for real estate or any similar tabular dataset.